# T3C: Temporal Transformer for Thermal Comfort — Walkthrough

This notebook demonstrates the complete T3C pipeline:
1. **Preprocessing** — Load, encode, and scale the ASHRAE II dataset
2. **PISSG** — Generate temporal sequences from static snapshots
3. **Training** — Train the Transformer model
4. **Evaluation** — Test accuracy, F1 scores, and confusion matrix

> **Paper**: *Transformer-Based Thermal Comfort Classification Using the ASHRAE II Dataset for Smart Building Applications*

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install category_encoders -q

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import numpy as np
import torch

from config import PISSGConfig, ModelConfig, TrainConfig
from src.preprocessing import load_and_preprocess
from src.pissg import create_geometric_sequential_data
from src.train import train_model, set_seed
from src.evaluate import evaluate_model, plot_training_history

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Preprocessing

Load the ASHRAE II dataset, apply cyclical season encoding, target-encode
categorical features, and scale all features to [0, 1].

In [ ]:
# Update this path to point to your dataset
DATA_PATH = '../data/df1.csv'

(
    X_train_scaled, X_val_scaled, X_test_scaled,
    y_train_enc, y_val_enc, y_test_enc,
    label_encoder, scaler, target_encoder,
) = load_and_preprocess(DATA_PATH)

print(f'\nClass names: {list(label_encoder.classes_)}')
print(f'Train: {X_train_scaled.shape}, Val: {X_val_scaled.shape}, Test: {X_test_scaled.shape}')

## 2. PISSG — Physics-Informed Synthetic Sequential Generation

Transform each static survey snapshot into a 12-step temporal sequence
(60 minutes of simulated environmental history at 5-minute intervals).

Geometric delta features (velocity vectors) are appended to capture the
*rate of change* in transient environmental variables.

In [ ]:
pissg_cfg = PISSGConfig(seq_length=12)

# Use different seeds for each split to avoid identical noise patterns
X_train_seq = create_geometric_sequential_data(X_train_scaled, pissg_cfg, seed=42)
X_val_seq   = create_geometric_sequential_data(X_val_scaled, pissg_cfg, seed=43)
X_test_seq  = create_geometric_sequential_data(X_test_scaled, pissg_cfg, seed=44)

print(f'Train sequences: {X_train_seq.shape}')   # (N, 12, 17)
print(f'Val sequences:   {X_val_seq.shape}')
print(f'Test sequences:  {X_test_seq.shape}')
print(f'Features: {X_train_scaled.shape[1]} base + {X_train_seq.shape[2] - X_train_scaled.shape[1]} deltas = {X_train_seq.shape[2]} total')

## 3. Training

Train the T3C Transformer with:
- Balanced class weights (handles ASHRAE II imbalance)
- Gradient clipping (stabilizes Transformer training)
- ReduceLROnPlateau scheduler
- Early stopping on validation loss

In [ ]:
model_cfg = ModelConfig(input_dim=X_train_seq.shape[2])
train_cfg = TrainConfig(epochs=55, batch_size=64)

model, history = train_model(
    X_train_seq, y_train_enc,
    X_val_seq, y_val_enc,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
)

In [ ]:
# Plot training curves
plot_training_history(history)

## 4. Evaluation

Evaluate on the held-out test set (completely unseen during training and
hyperparameter tuning). Reports per-class F1 scores and confusion matrix.

In [ ]:
class_names = list(label_encoder.classes_)

metrics = evaluate_model(
    model=model,
    X_test_seq=X_test_seq,
    y_test_enc=y_test_enc,
    class_names=class_names,
    device=device,
)

## 5. Save Model

Save the trained model weights and preprocessing artifacts for deployment.

In [ ]:
import joblib

torch.save(model.state_dict(), '../saved_models/t3c_best.pth')
joblib.dump(scaler, '../saved_models/scaler.pkl')
joblib.dump(label_encoder, '../saved_models/label_encoder.pkl')
joblib.dump(target_encoder, '../saved_models/target_encoder.pkl')

print('All artifacts saved to saved_models/')